In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from PIL import Image

import onnxruntime as ort

PROJECT_ROOT = Path.cwd().parent

EXPORT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "exports"
)

DEPLOYMENT_DIR = (
    PROJECT_ROOT
    / "deployment"
)

DEPLOYMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Inspectra — Deployment")
print("=" * 60)
print("Export directory:", EXPORT_DIR)
print("Deployment directory:", DEPLOYMENT_DIR)

Inspectra — Deployment
Export directory: d:\Inspectra\experiments\exports
Deployment directory: d:\Inspectra\deployment


In [2]:
MODEL_PATHS = {
    "bottle": (
        EXPORT_DIR
        / "bottle"
        / "bottle_finetuned.onnx"
    ),

    "pcb": (
        EXPORT_DIR
        / "pcb"
        / "pcb_finetuned.onnx"
    ),

    "road": (
        EXPORT_DIR
        / "road"
        / "road_resnet18_finetuned.onnx"
    ),
}

for name, path in MODEL_PATHS.items():

    print(
        f"{name:10}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

bottle    : FOUND
pcb       : FOUND
road      : FOUND


In [3]:
MODEL_METADATA = {
    "bottle": {
        "task": "detection",
        "input_size": 640,
        "classes": [
            "Cap",
            "Missing",
            "Wrong bottle",
            "box"
        ]
    },

    "pcb": {
        "task": "detection",
        "input_size": 640,
        "classes": [
            "mouse_bite",
            "spur",
            "missing_hole",
            "short",
            "open_circuit",
            "spurious_copper"
        ]
    },

    "road": {
        "task": "classification",
        "input_size": 224,
        "classes": [
            "Negative",
            "Positive"
        ]
    }
}

with open(
    DEPLOYMENT_DIR
    / "model_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        MODEL_METADATA,
        file,
        indent=4
    )

print("Metadata saved.")

Metadata saved.


In [4]:
def preprocess_image(
    image,
    size
):

    if isinstance(
        image,
        (str, Path)
    ):

        image = Image.open(
            image
        ).convert("RGB")

    elif isinstance(
        image,
        np.ndarray
    ):

        if image.ndim == 3:

            image = Image.fromarray(
                image
            ).convert("RGB")

        else:

            raise ValueError(
                "Expected RGB image."
            )

    elif isinstance(
        image,
        Image.Image
    ):

        image = image.convert(
            "RGB"
        )

    else:

        raise TypeError(
            "Unsupported image type."
        )

    image = image.resize(
        (size, size)
    )

    array = np.asarray(
        image,
        dtype=np.float32
    )

    array /= 255.0

    array = np.transpose(
        array,
        (2, 0, 1)
    )

    array = np.expand_dims(
        array,
        axis=0
    )

    return array

In [5]:
def inspect_model(
    model_path
):

    session = ort.InferenceSession(
        str(model_path),
        providers=[
            "CPUExecutionProvider"
        ]
    )

    print(
        "Model:",
        model_path
    )

    print(
        "Inputs:"
    )

    for item in session.get_inputs():

        print(
            item.name,
            item.shape,
            item.type
        )

    print(
        "Outputs:"
    )

    for item in session.get_outputs():

        print(
            item.name,
            item.shape,
            item.type
        )

    return session

In [6]:
sessions = {}

for name, path in MODEL_PATHS.items():

    if path.exists():

        sessions[name] = (
            inspect_model(path)
        )

        print("-" * 60)

Model: d:\Inspectra\experiments\exports\bottle\bottle_finetuned.onnx
Inputs:
images [1, 3, 640, 640] tensor(float)
Outputs:
output0 [1, 8, 8400] tensor(float)
------------------------------------------------------------
Model: d:\Inspectra\experiments\exports\pcb\pcb_finetuned.onnx
Inputs:
images [1, 3, 640, 640] tensor(float)
Outputs:
output0 [1, 10, 8400] tensor(float)
------------------------------------------------------------
Model: d:\Inspectra\experiments\exports\road\road_resnet18_finetuned.onnx
Inputs:
images ['batch', 3, 224, 224] tensor(float)
Outputs:
logits ['batch', 2] tensor(float)
------------------------------------------------------------


In [7]:
class ModelRegistry:

    def __init__(
        self,
        model_paths,
        metadata
    ):

        self.model_paths = (
            model_paths
        )

        self.metadata = (
            metadata
        )

        self.sessions = {}

    def load(
        self,
        name
    ):

        if name not in self.model_paths:

            raise ValueError(
                f"Unknown model: {name}"
            )

        if name not in self.sessions:

            path = self.model_paths[name]

            if not path.exists():

                raise FileNotFoundError(
                    str(path)
                )

            self.sessions[name] = (
                ort.InferenceSession(
                    str(path),
                    providers=[
                        "CPUExecutionProvider"
                    ]
                )
            )

        return self.sessions[name]

    def info(
        self,
        name
    ):

        if name not in self.metadata:

            raise ValueError(
                f"Unknown model: {name}"
            )

        return self.metadata[name]

    def list_models(
        self
    ):

        return list(
            self.model_paths.keys()
        )

In [8]:
registry = ModelRegistry(
    MODEL_PATHS,
    MODEL_METADATA
)

print(
    registry.list_models()
)

['bottle', 'pcb', 'road']


In [9]:
def sigmoid(
    x
):

    return 1.0 / (
        1.0 + np.exp(-x)
    )

In [10]:
def decode_yolo_output(
    output,
    classes,
    confidence_threshold=0.25
):

    predictions = output[0]

    if predictions.shape[0] < predictions.shape[1]:

        predictions = predictions.T

    num_classes = len(
        classes
    )

    boxes = []

    scores = []

    class_ids = []

    for prediction in predictions:

        if len(prediction) < (
            4 + num_classes
        ):

            continue

        box = prediction[:4]

        class_scores = (
            prediction[
                4:
                4 + num_classes
            ]
        )

        class_id = int(
            np.argmax(
                class_scores
            )
        )

        confidence = float(
            class_scores[class_id]
        )

        if confidence < confidence_threshold:

            continue

        boxes.append(
            box.tolist()
        )

        scores.append(
            confidence
        )

        class_ids.append(
            class_id
        )

    return {
        "boxes": boxes,
        "scores": scores,
        "class_ids": class_ids,
        "classes": [
            classes[i]
            for i in class_ids
        ]
    }

In [11]:
def predict_detection(
    model_name,
    image,
    confidence_threshold=0.25
):

    session = registry.load(
        model_name
    )

    metadata = registry.info(
        model_name
    )

    input_size = metadata[
        "input_size"
    ]

    tensor = preprocess_image(
        image,
        input_size
    )

    input_name = (
        session
        .get_inputs()[0]
        .name
    )

    start = time.perf_counter()

    outputs = session.run(
        None,
        {
            input_name: tensor
        }
    )

    latency_ms = (
        time.perf_counter()
        - start
    ) * 1000

    decoded = (
        decode_yolo_output(
            outputs[0],
            metadata["classes"],
            confidence_threshold
        )
    )

    decoded[
        "model"
    ] = model_name

    decoded[
        "task"
    ] = "detection"

    decoded[
        "latency_ms"
    ] = latency_ms

    return decoded

In [12]:
def softmax(
    x
):

    x = x - np.max(
        x,
        axis=1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        / np.sum(
            exp_x,
            axis=1,
            keepdims=True
        )
    )

In [13]:
def predict_classification(
    model_name,
    image
):

    session = registry.load(
        model_name
    )

    metadata = registry.info(
        model_name
    )

    tensor = preprocess_image(
        image,
        metadata["input_size"]
    )

    input_name = (
        session
        .get_inputs()[0]
        .name
    )

    start = time.perf_counter()

    outputs = session.run(
        None,
        {
            input_name: tensor
        }
    )

    latency_ms = (
        time.perf_counter()
        - start
    ) * 1000

    logits = outputs[0]

    probabilities = softmax(
        logits
    )[0]

    class_id = int(
        np.argmax(
            probabilities
        )
    )

    return {
        "model": model_name,
        "task": "classification",
        "class_id": class_id,
        "class": metadata[
            "classes"
        ][class_id],
        "confidence": float(
            probabilities[class_id]
        ),
        "probabilities": {
            name: float(
                probabilities[index]
            )
            for index, name
            in enumerate(
                metadata["classes"]
            )
        },
        "latency_ms": latency_ms
    }

In [14]:
def predict(
    model_name,
    image,
    confidence_threshold=0.25
):

    metadata = registry.info(
        model_name
    )

    if metadata["task"] == "detection":

        return predict_detection(
            model_name,
            image,
            confidence_threshold
        )

    if metadata["task"] == "classification":

        return predict_classification(
            model_name,
            image
        )

    raise ValueError(
        f"Unsupported task: "
        f"{metadata['task']}"
    )

In [16]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
)

print(PROCESSED_DATA)

bottle_test_image = next(
    (
        PROCESSED_DATA
        / "bottle"
        / "test"
        / "images"
    ).iterdir()
)

bottle_prediction = predict(
    "bottle",
    bottle_test_image
)

print(
    json.dumps(
        bottle_prediction,
        indent=4
    )
)

d:\Inspectra\datasets\processed
{
    "boxes": [
        [
            202.1424560546875,
            440.75543212890625,
            73.42669677734375,
            134.2911376953125
        ],
        [
            202.12167358398438,
            441.10125732421875,
            73.35881042480469,
            134.543212890625
        ],
        [
            202.07347106933594,
            440.94110107421875,
            73.6461181640625,
            133.77618408203125
        ],
        [
            202.22286987304688,
            441.2783203125,
            73.08692932128906,
            134.13375854492188
        ],
        [
            202.1984405517578,
            441.4285583496094,
            73.43157958984375,
            134.24127197265625
        ],
        [
            202.01031494140625,
            441.41729736328125,
            73.56109619140625,
            133.47763061523438
        ],
        [
            202.00384521484375,
            441.47491455078125,
      

In [17]:
pcb_test_image = next(
    (
        PROCESSED_DATA
        / "pcb"
        / "test"
        / "images"
    ).iterdir()
)

pcb_prediction = predict(
    "pcb",
    pcb_test_image
)

print(
    json.dumps(
        pcb_prediction,
        indent=4
    )
)

{
    "boxes": [],
    "scores": [],
    "class_ids": [],
    "classes": [],
    "model": "pcb",
    "task": "detection",
    "latency_ms": 25.18749999580905
}


In [18]:
road_test_image = next(
    (
        PROCESSED_DATA
        / "road"
        / "test"
        / "Negative"
    ).iterdir()
)

road_prediction = predict(
    "road",
    road_test_image
)

print(
    json.dumps(
        road_prediction,
        indent=4
    )
)

{
    "model": "road",
    "task": "classification",
    "class_id": 0,
    "class": "Negative",
    "confidence": 0.9999996423721313,
    "probabilities": {
        "Negative": 0.9999996423721313,
        "Positive": 3.4226636103085184e-07
    },
    "latency_ms": 9.31049999780953
}


In [1]:
import subprocess
import sys

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "deployment.api:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ]
)

print(
    "Inspectra API started on "
    "http://127.0.0.1:8000"
)

Inspectra API started on http://127.0.0.1:8000
